# EDA — Amazon Computers

Exploratory analysis of the Amazon Co-Buy Computer node classification graph. Self-contained: the dataset loader is copied inline below (not imported from `common/data.py`), so this one file runs on its own — copy it out of the repo and it still works. Evaluated against the fixed `split_idx.csv` split when present, so the numbers here match what the benchmark's training scripts actually train/evaluate on.

In [ ]:
# Fresh environment (Kaggle / Colab / a clean venv) that doesn't have these yet?
# Uncomment and run once — everything below only needs these, nothing else from this repo.
# %pip install -q torch numpy scipy pandas matplotlib
# %pip install -q torch-geometric  # optional: only used to auto-download the dataset if data/ is empty

## 1. Setup

In [ ]:
import json
import logging
from pathlib import Path
from types import SimpleNamespace

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy.sparse as sp
import torch

try:
    from IPython.display import display
except ImportError:
    def display(x):
        print(x)

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

logging.basicConfig(level=logging.INFO, format="%(message)s")
logger = logging.getLogger("eda")

OUT_DIR = REPO_ROOT / "outputs" / "eda"
OUT_DIR.mkdir(parents=True, exist_ok=True)

Dataset loader, copied inline from `common/data.py` rather than imported — this is what keeps the notebook runnable standalone.

In [ ]:
def build_stratified_split(labels, train_ratio=0.6, valid_ratio=0.2, seed=42):
    rng = np.random.default_rng(seed)
    labels_np = labels.cpu().numpy()
    split_parts = {"train": [], "valid": [], "test": []}
    for cls in np.unique(labels_np):
        cls_idx = np.where(labels_np == cls)[0]
        rng.shuffle(cls_idx)
        n_total = len(cls_idx)
        n_train = int(n_total * train_ratio)
        n_valid = int(n_total * valid_ratio)
        if n_train + n_valid >= n_total and n_total >= 3:
            n_train = max(1, n_total - 2)
            n_valid = 1
        elif n_train + n_valid >= n_total and n_total == 2:
            n_train = 1
            n_valid = 1
        elif n_total == 1:
            n_train = 1
            n_valid = 0
        split_parts["train"].append(cls_idx[:n_train])
        split_parts["valid"].append(cls_idx[n_train:n_train + n_valid])
        split_parts["test"].append(cls_idx[n_train + n_valid:])
    split_idx = {}
    for name, parts in split_parts.items():
        idx = np.concatenate([p for p in parts if len(p) > 0]) if any(len(p) > 0 for p in parts) else np.array([], dtype=np.int64)
        rng.shuffle(idx)
        split_idx[name] = torch.from_numpy(idx).long()
    return split_idx


def find_amazon_computer_npz(root):
    root = Path(root)
    candidates = sorted(root.glob("amazon_co_buy_computer*/amazon_co_buy_computer.npz"))
    candidates += sorted(root.glob("**/amazon_co_buy_computer.npz"))
    if not candidates:
        try:
            from torch_geometric.datasets import Amazon
            Amazon(root=str(root), name="Computers")
        except Exception as e:
            raise FileNotFoundError(
                f"amazon_co_buy_computer.npz not found under {root} and auto-download failed: {e}\n"
                "Place the file manually, or `pip install torch-geometric` to auto-download."
            ) from e
        candidates = sorted(root.glob("amazon_co_buy_computer*/amazon_co_buy_computer.npz"))
        candidates += sorted(root.glob("**/amazon_co_buy_computer.npz"))
    if not candidates:
        raise FileNotFoundError(f"amazon_co_buy_computer.npz not found under {root}")
    return candidates[0]


def load_products(root, logger, split_seed=42):
    npz_path = find_amazon_computer_npz(root)
    with np.load(npz_path, allow_pickle=True) as loader:
        loader = dict(loader)
    adj = sp.csr_matrix((loader["adj_data"], loader["adj_indices"], loader["adj_indptr"]), shape=tuple(loader["adj_shape"]))
    adj = adj.maximum(adj.T).tocoo()
    edge_index = torch.from_numpy(np.vstack([adj.row, adj.col]).astype(np.int64, copy=False))
    if "attr_data" in loader:
        x = sp.csr_matrix((loader["attr_data"], loader["attr_indices"], loader["attr_indptr"]), shape=tuple(loader["attr_shape"])).toarray()
    else:
        x = loader["attr_matrix"]
    labels = torch.as_tensor(loader["labels"], dtype=torch.long).view(-1)
    num_classes = int(len(np.unique(loader["labels"])))
    split_idx = build_stratified_split(labels, seed=split_seed)
    data = SimpleNamespace(x=torch.as_tensor(x, dtype=torch.float32), edge_index=edge_index.long(), num_nodes=int(labels.numel()))
    logger.info(
        "Loaded Amazon Computers from %s | nodes=%d edges=%d train=%d valid=%d test=%d classes=%d",
        npz_path, data.num_nodes, data.edge_index.size(1),
        split_idx["train"].numel(), split_idx["valid"].numel(), split_idx["test"].numel(), num_classes,
    )
    return data, labels, split_idx, num_classes


def load_split_idx_csv(path):
    parts = {"train": [], "valid": [], "test": []}
    with open(path, newline="", encoding="utf-8") as f:
        next(f)
        for line in f:
            line = line.strip()
            if not line:
                continue
            split, idx = line.split(",", 1)
            parts[split].append(int(idx))
    return {k: torch.tensor(v, dtype=torch.long) for k, v in parts.items()}

## 2. Load dataset + fixed split

In [ ]:
data, labels, split_idx, num_classes = load_products(REPO_ROOT / "data", logger)

split_file = REPO_ROOT / "split_idx.csv"
if split_file.is_file():
    split_idx = load_split_idx_csv(split_file)
    logger.info("Using fixed split from %s", split_file)

num_nodes = data.num_nodes
num_edges = data.edge_index.size(1)  # directed edge count (graph is symmetrised, so this is 2x the undirected count)
num_features = data.x.size(1)
split_sizes = {k: int(v.numel()) for k, v in split_idx.items()}

summary = {
    "num_nodes": int(num_nodes),
    "num_edges_directed": int(num_edges),
    "num_node_features": int(num_features),
    "num_classes": int(num_classes),
    "split_sizes": split_sizes,
}
print(json.dumps(summary, indent=2))
(OUT_DIR / "dataset_summary.json").write_text(json.dumps(summary, indent=2))

## 3. Split sizes

In [ ]:
split_df = pd.DataFrame([
    {"split": k, "nodes": v, "percent": 100 * v / num_nodes}
    for k, v in split_sizes.items()
])
display(split_df)

ax = split_df.plot(kind="bar", x="split", y="nodes", legend=False, title="Split sizes", figsize=(6, 4))
ax.set_ylabel("nodes")
plt.tight_layout()
plt.savefig(OUT_DIR / "split_sizes.png", dpi=150)
plt.show()

## 4. Label distribution

In [ ]:
label_counts = torch.bincount(labels, minlength=num_classes).numpy()
label_df = pd.DataFrame({"class": np.arange(num_classes), "count": label_counts})
label_df["percent"] = 100 * label_df["count"] / label_df["count"].sum()
display(label_df.sort_values("count", ascending=False))
label_df.to_csv(OUT_DIR / "label_distribution.csv", index=False)

rows = []
for split_name, idx in split_idx.items():
    counts = torch.bincount(labels[idx], minlength=num_classes).numpy()
    for cls, count in enumerate(counts):
        rows.append({"split": split_name, "class": cls, "count": int(count)})
split_label_df = pd.DataFrame(rows)
pivot = split_label_df.pivot(index="class", columns="split", values="count")

pivot.plot(kind="bar", figsize=(12, 4.5), title="Class distribution by split")
plt.xlabel("class")
plt.ylabel("nodes")
plt.tight_layout()
plt.savefig(OUT_DIR / "split_label_distribution.png", dpi=150)
plt.show()

## 5. Feature statistics

Node features are bag-of-words counts over product reviews — expect a sparse, right-skewed distribution.

In [ ]:
x = data.x
feature_stats = {
    "mean": float(x.mean()),
    "std": float(x.std()),
    "min": float(x.min()),
    "max": float(x.max()),
    "zero_fraction": float((x == 0).float().mean()),
}
print(json.dumps(feature_stats, indent=2))
(OUT_DIR / "feature_stats.json").write_text(json.dumps(feature_stats, indent=2))

sample_idx = torch.randperm(num_nodes)[:min(5000, num_nodes)]
plt.figure(figsize=(7, 4))
plt.hist(x[sample_idx].flatten().numpy(), bins=80)
plt.title(f"Feature value histogram (sampled {sample_idx.numel():,} nodes)")
plt.xlabel("feature value")
plt.ylabel("frequency")
plt.yscale("log")
plt.tight_layout()
plt.savefig(OUT_DIR / "feature_value_histogram.png", dpi=150)
plt.show()

## 6. Degree statistics

In [ ]:
row, col = data.edge_index
degree = torch.bincount(row, minlength=num_nodes)

degree_stats = {
    "mean": float(degree.float().mean()),
    "median": float(degree.float().median()),
    "max": int(degree.max()),
    "isolated_nodes": int((degree == 0).sum()),
}
print(json.dumps(degree_stats, indent=2))
(OUT_DIR / "degree_stats.json").write_text(json.dumps(degree_stats, indent=2))

degree_df = pd.DataFrame({"node": np.arange(num_nodes), "degree": degree.numpy(), "label": labels.numpy()})

plt.figure(figsize=(8, 4))
plt.hist(torch.log1p(degree.float()).numpy(), bins=80)
plt.title("log1p(degree) distribution")
plt.xlabel("log1p(degree)")
plt.ylabel("nodes")
plt.tight_layout()
plt.savefig(OUT_DIR / "degree_distribution.png", dpi=150)
plt.show()

degree_by_class = degree_df.groupby("label")["degree"].agg(["mean", "median", "max", "count"]).reset_index()
display(degree_by_class.sort_values("mean", ascending=False))

## 7. Homophily

Fraction of edges whose endpoints share the same label — a quick sanity check for how much a graph-aware model *should* be able to exploit over a feature-only baseline (MLP).

In [ ]:
same_label = (labels[row] == labels[col]).float()
homophily = float(same_label.mean())
print(f"Edge homophily: {homophily:.4f} over {row.numel():,} directed edges")
(OUT_DIR / "homophily.json").write_text(json.dumps({"edges": int(row.numel()), "homophily": homophily}, indent=2))

## Notes

- Every training script under `graphsage/`, `sagat/`, `gamlp/`, `lightgcn/` defaults to `--split-file split_idx.csv` (the same split loaded above), so results are directly comparable across models.
- See `notebooks/02_train_all_standalone.ipynb` to train every model and compare results in one run.